# CT-RATE → official EXACT Y-Mamba image encodings → Google Drive

This notebook creates **EXACT image features for the full 4,385-pair longitudinal
CT-RATE cohort**. It does **not** use CT-CLIP to encode images. The CT-CLIP cache is
read only for a coverage comparison; the target set is reconstructed from CT-RATE's
official report and metadata CSVs.

For each target volume it:

1. downloads one gated CT-RATE NIfTI from Hugging Face;
2. preprocesses it into a configurable 3D tensor;
3. runs the **official `JasonW375/EXACT` `YMamba` model** loaded from
   `ymamba_pretrain_best.pth`;
4. extracts `encoder5(vit(x)[3])`, globally pools it to an **official-encoder-derived
   768-d image vector**, and saves it immediately to Drive;
5. deletes the raw volume and continues.

The loop is resumable and processes validation before training.

## Does EXACT have image and text encoders?

**EXACT has a 3D image encoder (Y-Mamba), not a CLIP-style paired text encoder.**
Reports supply weak disease labels during pretraining, but EXACT does not produce text
vectors in the same space as its image features. The saved vectors here are image-only.

## Reproducibility caveat

The public EXACT README references `EXACT_Pretrain/data_preprocessed/*.py`, but those
raw-NIfTI preprocessing scripts are absent from the current public repository. This
notebook therefore uses the **exact public architecture and exact checkpoint**, while
making the reconstructed NIfTI preprocessing explicit and recording it in every output.
Do not describe preprocessing as bit-exact to the paper unless the authors provide the
missing scripts.

**Required Colab runtime:** before running any cell, open **Runtime → Change runtime
type**, select an NVIDIA GPU, and set **Runtime Version → 2025.07**. That past runtime
provides Python 3.11 and PyTorch 2.6, for which matching prebuilt Mamba CUDA wheels exist.
A100/L4 (24 GB+) is recommended.

In [ ]:
# GPU/runtime check. PyTorch 2.4.1 has no Python 3.13 wheel, so it cannot be installed
# into the current default Colab runtime. Use Colab's 2025.07 past runtime instead.
!nvidia-smi || echo 'NO GPU — Runtime > Change runtime type > GPU'
import platform, sys, torch
print('python:', platform.python_version())
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda,
      '| available:', torch.cuda.is_available())
assert sys.version_info[:2] == (3, 11) and torch.__version__.split('+')[0] == '2.6.0', (
    f'Unsupported Colab image: Python {platform.python_version()}, torch {torch.__version__}.\n'
    'Do not try to pip-install torch 2.4.1 under Python 3.13; no such wheel exists.\n'
    'Select Runtime > Change runtime type > Runtime Version > 2025.07, choose a GPU, '
    'save, and then use Runtime > Run all.'
)
assert torch.cuda.is_available(), 'A CUDA GPU is required for full-scale EXACT encoding.'

## 1. Install the official EXACT foundation-model environment

EXACT documents PyTorch 2.4.1 + CUDA 12.1, but that PyTorch release has no Python 3.13
wheel. The current default Colab runtime therefore cannot be repaired by pip-downgrading
PyTorch. This notebook uses Colab's **2025.07 past runtime** (Python 3.11 / PyTorch 2.6)
and downloads matching prebuilt CUDA wheels directly. It never compiles Mamba from source.

Colab keeps past runtime versions for one year. If `2025.07` is no longer offered, stop:
the wheel matrix below must be updated and smoke-tested for a newer runtime rather than
silently compiling or mixing binary-incompatible packages.

In [ ]:
# Deliberately do not replace Colab's core PyTorch package. Compiled extension wheels
# below are selected for the 2025.07 runtime's Python and PyTorch versions.
import platform, sys, torch
assert sys.version_info[:2] == (3, 11), platform.python_version()
assert torch.__version__.split('+')[0] == '2.6.0', torch.__version__
print('Supported base runtime:', platform.python_version(), torch.__version__, torch.version.cuda)

In [ ]:
# Install binary-compatible prebuilt Mamba wheels (no local CUDA compilation).
%cd /content
![ -d EXACT/.git ] || git clone --depth 1 https://github.com/JasonW375/EXACT.git EXACT

import os, subprocess, sys, torch
from pathlib import Path

assert torch.__version__.split('+')[0] == '2.6.0', 'Select Colab Runtime Version 2025.07.'
assert sys.version_info[:2] == (3, 11), 'Select Colab Runtime Version 2025.07.'

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ninja', 'packaging', 'einops==0.8.0', 'transformers',
    'huggingface_hub', 'nibabel==5.3.2', 'scipy', 'tqdm', 'gdown', 'monai==1.3.0'
], check=True)

py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
abi = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
platform_tag = 'linux_x86_64'
causal_name = f'causal_conv1d-1.5.0.post8+cu12torch2.6cxx11abi{abi}-{py_tag}-{py_tag}-{platform_tag}.whl'
mamba_name = f'mamba_ssm-2.2.4+cu12torch2.6cxx11abi{abi}-{py_tag}-{py_tag}-{platform_tag}.whl'
causal_url = 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/' + causal_name.replace('+', '%2B')
mamba_url = 'https://github.com/state-spaces/mamba/releases/download/v2.2.4/' + mamba_name.replace('+', '%2B')

print('Python tag:', py_tag, '| CXX11 ABI:', abi)
print('Installing prebuilt causal-conv1d wheel:', causal_name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', causal_url], check=True)
print('Installing prebuilt mamba-ssm wheel:', mamba_name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', mamba_url], check=True)

import causal_conv1d, mamba_ssm
print('causal_conv1d:', causal_conv1d.__version__)
print('mamba_ssm:', mamba_ssm.__version__)

EXACT_ROOT = '/content/EXACT'
EXACT_PRETRAIN = os.path.join(EXACT_ROOT, 'EXACT_Pretrain')
if EXACT_PRETRAIN not in sys.path:
    sys.path.insert(0, EXACT_PRETRAIN)
from models.ymamba.ymamba import YMamba
print('Imported official YMamba from:', sys.modules[YMamba.__module__].__file__)

## 2. Mount Drive and configure paths

EXACT results are kept under `MyDrive/3dCT/exact_cache/pooled/`. If present,
`MyDrive/3dCT/ctclip_cache/img/*.pt` is used only for a coverage comparison.

The notebook automatically downloads the public official checkpoint to:

`MyDrive/3dCT/exact_weights/ymamba_pretrain_best.pth`

If automatic downloading is blocked by Google Drive quota, manually download
`01_pretrain/ymamba_pretrain_best.pth` from the official folder and place it there:
https://drive.google.com/drive/folders/1i2J6XUqTm2G8m3-OlbH7Wt00aBxIpClf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, json, os, shutil, time
from pathlib import Path

DRIVE_3DCT = Path('/content/drive/MyDrive/3dCT')
CTCLIP_IMG_DIR = DRIVE_3DCT / 'ctclip_cache' / 'img'
EXACT_CACHE = DRIVE_3DCT / 'exact_cache'
EXACT_POOLED_DIR = EXACT_CACHE / 'pooled'
EXACT_SPATIAL_DIR = EXACT_CACHE / 'stage4_spatial'
EXACT_MANIFEST_DIR = EXACT_CACHE / 'manifests'
CHECKPOINT_PATH = DRIVE_3DCT / 'exact_weights' / 'ymamba_pretrain_best.pth'
TMP = Path('/content/_exact_volume_tmp')

for p in [EXACT_POOLED_DIR, EXACT_MANIFEST_DIR, CHECKPOINT_PATH.parent, TMP]:
    p.mkdir(parents=True, exist_ok=True)

# Full-scale defaults. The longitudinal manifest determines targets; there is no volume limit.
PROCESS_VALID_FIRST = True
SAVE_STAGE4_SPATIAL = False  # much larger cache; pooled 768-d is always saved
USE_AMP = True

# Reconstructed raw-NIfTI preprocessing. The missing official scripts prevent a
# bit-exact claim. Keep these fixed across all volumes in an experiment.
TARGET_DHW = (64, 64, 64)  # matches official config block_size=64; 16x downsampling
HU_MIN, HU_MAX = -1000.0, 1000.0
NORMALIZE_TO = 'zero_one'
PREPROCESSING_ID = 'reconstructed_ctrate_canonical_resize64_hu-1000_1000_zero_one_v1'

print('CT-CLIP inventory:', CTCLIP_IMG_DIR)
print('EXACT output:', EXACT_CACHE)
print('Checkpoint:', CHECKPOINT_PATH)

## 3. Official checkpoint

The authors publish the checkpoint in a public Google Drive folder. This cell downloads
only `01_pretrain/ymamba_pretrain_best.pth` (published size: 998.2 MB) and persists it to
your Drive. It stages the download locally and uses a `.part` destination so an interrupted
copy is never mistaken for a valid checkpoint. Existing valid files are reused.

In [ ]:
import gdown

CHECKPOINT_FILE_ID = '1j6YyW1pvOGymck4OVJsxSfNrtFivXlFE'
MIN_CHECKPOINT_BYTES = 800 * 1024 * 1024
checkpoint_ok = CHECKPOINT_PATH.exists() and CHECKPOINT_PATH.stat().st_size >= MIN_CHECKPOINT_BYTES

if not checkpoint_ok:
    local_download = TMP / 'ymamba_pretrain_best.pth.download'
    drive_partial = CHECKPOINT_PATH.with_suffix(CHECKPOINT_PATH.suffix + '.part')
    local_download.unlink(missing_ok=True)
    drive_partial.unlink(missing_ok=True)

    print('Downloading the official 998.2 MB EXACT checkpoint...')
    try:
        result = gdown.download(id=CHECKPOINT_FILE_ID, output=str(local_download), quiet=False)
        if result is None or not local_download.exists():
            raise RuntimeError('gdown did not produce a checkpoint file.')
        downloaded_bytes = local_download.stat().st_size
        if downloaded_bytes < MIN_CHECKPOINT_BYTES:
            raise RuntimeError(
                f'Download is only {downloaded_bytes / 1024**2:.1f} MiB; expected about 998.2 MB. '
                'Google Drive may have returned a quota or access page.')

        print('Copying validated download to Google Drive...')
        shutil.copyfile(local_download, drive_partial)
        if drive_partial.stat().st_size != downloaded_bytes:
            raise IOError('Drive copy size does not match the local download.')
        drive_partial.replace(CHECKPOINT_PATH)
    except Exception as exc:
        drive_partial.unlink(missing_ok=True)
        raise RuntimeError(
            'Automatic checkpoint download failed. Manually download '
            '01_pretrain/ymamba_pretrain_best.pth from:\n'
            'https://drive.google.com/drive/folders/1I5OHU0_QMpxqRMPNyxjeYPSZjDpRhuRH\n'
            f'and place it at {CHECKPOINT_PATH}. Original error: {exc}'
        ) from exc
    finally:
        local_download.unlink(missing_ok=True)

size_gb = CHECKPOINT_PATH.stat().st_size / 1024**3
assert CHECKPOINT_PATH.stat().st_size >= MIN_CHECKPOINT_BYTES, (
    f'Checkpoint is unexpectedly small ({size_gb:.2f} GiB): {CHECKPOINT_PATH}')
print(f'Official checkpoint present: {size_gb:.2f} GiB')

## 4. Reconstruct the full longitudinal CT-RATE target set

The `valid_*` prefix denotes CT-RATE's source validation partition (3,039 reconstructed
volumes), not the longitudinal validation examples. We reproduce the project's official
report/metadata pairing procedure and select only the unique prior/current volumes needed
by all 4,385 dated longitudinal pairs: 6,762 `train_*` and 429 `valid_*` volumes.

This downloads only the four official CSVs (~101 MB), not any CT volume. The resulting
compact pair and target manifests are saved to Drive. The CT-CLIP cache is audited for
comparison but does not determine the EXACT target set.

In [ ]:
import csv
from collections import defaultdict
from datetime import datetime
from huggingface_hub import login, hf_hub_download

login()  # paste a Hugging Face READ token after accepting the CT-RATE terms
HF_TOKEN = os.environ.get('HF_TOKEN') or True
CTRATE_REPO = 'ibrahimhamamci/CT-RATE'
CSV_CACHE = Path('/content/_ctrate_csv_cache')
CSV_CACHE.mkdir(parents=True, exist_ok=True)

REPORT_CSVS = [
    'dataset/radiology_text_reports/train_reports.csv',
    'dataset/radiology_text_reports/validation_reports.csv',
]
META_CSVS = [
    'dataset/metadata/train_metadata.csv',
    'dataset/metadata/validation_metadata.csv',
]

def fetch_csv(remote_path):
    return hf_hub_download(
        CTRATE_REPO, remote_path, repo_type='dataset', token=HF_TOKEN,
        local_dir=str(CSV_CACHE), force_download=False)

def parse_volume_name(volume):
    base = volume.removesuffix('.nii.gz').removesuffix('.nii')
    parts = base.split('_')
    if len(parts) < 4:
        return None
    return parts[0], parts[1], parts[2], parts[3]  # split, patient, scan, reconstruction

def parse_study_date(value):
    value = (value or '').strip()
    for fmt, width in (('%Y%m%d', 8), ('%Y-%m-%d', 10)):
        try:
            return datetime.strptime(value[:width], fmt)
        except (TypeError, ValueError):
            pass
    return None

# Reports define the released volume inventory; metadata supplies real study dates.
report_volumes = set()
for remote_path in REPORT_CSVS:
    with open(fetch_csv(remote_path), newline='', encoding='utf-8', errors='ignore') as f:
        for row in csv.DictReader(f):
            volume = (row.get('VolumeName') or '').strip()
            if volume:
                report_volumes.add(volume)

study_dates = {}
for remote_path in META_CSVS:
    with open(fetch_csv(remote_path), newline='', encoding='utf-8', errors='ignore') as f:
        for row in csv.DictReader(f):
            volume = (row.get('VolumeName') or '').strip()
            if volume:
                study_dates[volume] = parse_study_date(row.get('StudyDate'))

# One representative reconstruction per study: prefer one with a date, then the
# lexicographically smallest volume name. This exactly mirrors 12_ctrate_enrich.py.
studies = defaultdict(dict)
for volume in report_volumes:
    parsed = parse_volume_name(volume)
    if parsed is None:
        continue
    split, patient_id, scan_id, _ = parsed
    patient = f'{split}_{patient_id}'
    candidate = {'volume': volume, 'date': study_dates.get(volume)}
    current = studies[patient].get(scan_id)
    if current is None:
        studies[patient][scan_id] = candidate
    else:
        candidate_has_date = candidate['date'] is not None
        current_has_date = current['date'] is not None
        if ((candidate_has_date and not current_has_date) or
            (candidate_has_date == current_has_date and volume < current['volume'])):
            studies[patient][scan_id] = candidate

pair_rows = []
for patient, scan_map in studies.items():
    if len(scan_map) < 2:
        continue
    ordered = sorted(
        scan_map.items(),
        key=lambda item: (item[1]['date'] or datetime.max, item[0]),
    )
    for (_, prior), (_, current) in zip(ordered, ordered[1:]):
        if prior['date'] is None or current['date'] is None:
            continue
        delta_days = (current['date'] - prior['date']).days
        if delta_days <= 0:
            continue
        pair_rows.append({
            'patient': patient,
            'prior_volume': prior['volume'],
            'curr_volume': current['volume'],
            'prior_date': prior['date'].strftime('%Y-%m-%d'),
            'curr_date': current['date'].strftime('%Y-%m-%d'),
            'delta_days': delta_days,
        })

pair_rows.sort(key=lambda row: (row['patient'], row['prior_date'], row['curr_date']))
train_pairs = [row for row in pair_rows if row['patient'].startswith('train_')]
valid_pairs = [row for row in pair_rows if row['patient'].startswith('valid_')]

def unique_pair_volumes(rows):
    return sorted({row[key] for row in rows for key in ('prior_volume', 'curr_volume')})

train_targets = unique_pair_volumes(train_pairs)
valid_targets = unique_pair_volumes(valid_pairs)

EXPECTED_COUNTS = {
    'pairs': 4385, 'train_pairs': 4125, 'valid_pairs': 260,
    'train_volumes': 6762, 'valid_volumes': 429, 'total_volumes': 7191,
}
actual_counts = {
    'pairs': len(pair_rows), 'train_pairs': len(train_pairs), 'valid_pairs': len(valid_pairs),
    'train_volumes': len(train_targets), 'valid_volumes': len(valid_targets),
    'total_volumes': len(set(train_targets) | set(valid_targets)),
}
assert actual_counts == EXPECTED_COUNTS, (
    'Official CSVs no longer reproduce the reviewed longitudinal manifest. '
    f'Expected {EXPECTED_COUNTS}, got {actual_counts}. Stop and audit upstream changes.')
assert not (set(train_targets) & set(valid_targets)), 'Train/valid volume overlap.'

pair_manifest_path = EXACT_MANIFEST_DIR / 'longitudinal_pairs.csv'
with open(pair_manifest_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(pair_rows[0]))
    writer.writeheader()
    writer.writerows(pair_rows)

ctclip_cached = {p.stem + '.nii.gz' for p in CTCLIP_IMG_DIR.glob('*.pt')}
required = set(train_targets) | set(valid_targets)
ctclip_missing = sorted(required - ctclip_cached)
ctclip_extra = sorted(ctclip_cached - required)

targets_payload = {
    'created_unix': time.time(),
    'source': REPORT_CSVS + META_CSVS,
    'selection': 'dated consecutive longitudinal pairs, one reconstruction per study',
    **actual_counts,
    'ctclip_present': len(required & ctclip_cached),
    'ctclip_missing': len(ctclip_missing),
    'ctclip_extra': len(ctclip_extra),
    'train': train_targets,
    'valid': valid_targets,
    'missing_from_ctclip_cache': ctclip_missing,
}
with open(EXACT_MANIFEST_DIR / 'target_volumes.json', 'w') as f:
    json.dump(targets_payload, f, indent=2)

print('Full longitudinal CT-RATE targets')
print('  pairs:', len(pair_rows), '| train_*:', len(train_pairs), '| valid_*:', len(valid_pairs))
print('  unique volumes:', len(required), '| train_*:', len(train_targets), '| valid_*:', len(valid_targets))
print('CT-CLIP coverage:', len(required & ctclip_cached), 'present,',
      len(ctclip_missing), 'missing,', len(ctclip_extra), 'unrelated cache entries')
print('Saved pair manifest:', pair_manifest_path)

## 5. Authenticate to gated CT-RATE and define disk-safe downloads

Accept the CT-RATE terms on Hugging Face first. Each raw NIfTI is downloaded into a
temporary directory that is completely removed after the encoding attempt, including
Hugging Face's local metadata/cache files.

In [ ]:
def remote_candidates(volume):
    base = volume.removesuffix('.nii.gz').removesuffix('.nii')
    split, patient_id, scan = base.split('_')[:3]
    patient = f'{split}_{patient_id}'
    scan_folder = f'{split}_{patient_id}_{scan}'
    folders = [f'{split}_fixed', split] if split in {'train', 'valid'} else [split]
    return [f'dataset/{folder}/{patient}/{scan_folder}/{volume}' for folder in folders]

def reset_tmp():
    shutil.rmtree(TMP, ignore_errors=True)
    TMP.mkdir(parents=True, exist_ok=True)

def download_volume(volume):
    reset_tmp()
    errors = []
    for remote_path in remote_candidates(volume):
        try:
            return hf_hub_download(
                CTRATE_REPO, remote_path, repo_type='dataset', token=HF_TOKEN,
                local_dir=str(TMP), force_download=False)
        except Exception as exc:
            errors.append(f'{remote_path}: {type(exc).__name__}')
    print('DOWNLOAD FAIL', volume, '|', ' ; '.join(errors))
    return None

print('Example candidates:', remote_candidates(valid_targets[0]))

## 6. Reconstructed NIfTI preprocessing

The operation is deterministic:

- canonicalize orientation with nibabel;
- read HU as float32;
- clip to `[HU_MIN, HU_MAX]`;
- resize the complete canonical volume to `TARGET_DHW` with trilinear interpolation;
- scale to `[0,1]`;
- return `[1,1,D,H,W]`.

This is deliberately isolated in one function so it can be replaced if the authors
release the missing official scripts. Changing it after starting a cache requires a new
`PREPROCESSING_ID` and output directory.

In [ ]:
import nibabel as nib
import numpy as np
import torch.nn.functional as F

def preprocess_exact_nifti(path):
    nii = nib.load(path)
    original_shape = tuple(int(x) for x in nii.shape[:3])
    original_axcodes = tuple(str(x) for x in nib.aff2axcodes(nii.affine))
    canonical = nib.as_closest_canonical(nii)
    canonical_axcodes = tuple(str(x) for x in nib.aff2axcodes(canonical.affine))
    xyz = canonical.get_fdata(dtype=np.float32)
    if xyz.ndim != 3:
        raise ValueError(f'Expected 3D NIfTI, got {xyz.shape}')
    xyz = np.nan_to_num(xyz, nan=HU_MIN, posinf=HU_MAX, neginf=HU_MIN)
    xyz = np.clip(xyz, HU_MIN, HU_MAX)

    # nibabel array is X,Y,Z; model tensor is D,H,W = Z,Y,X.
    dhw = np.ascontiguousarray(xyz.transpose(2, 1, 0))
    x = torch.from_numpy(dhw).unsqueeze(0).unsqueeze(0)
    x = F.interpolate(x, size=TARGET_DHW, mode='trilinear', align_corners=False)
    x = (x - HU_MIN) / (HU_MAX - HU_MIN)
    x = x.contiguous().float()
    if not torch.isfinite(x).all():
        raise ValueError('Non-finite values after preprocessing.')
    metadata = {
        'original_shape_xyz': original_shape,
        'original_axcodes': original_axcodes,
        'canonical_shape_xyz': tuple(int(v) for v in canonical.shape[:3]),
        'canonical_axcodes': canonical_axcodes,
        'model_shape_bcdhw': tuple(int(v) for v in x.shape),
        'target_dhw': TARGET_DHW,
        'hu_clip': (HU_MIN, HU_MAX),
        'normalization': NORMALIZE_TO,
        'preprocessing_id': PREPROCESSING_ID,
    }
    return x, metadata

## 7. Build official Y-Mamba, load weights, and expose encoder features

The model is the official `YMamba` defaults: feature channels `48/96/192/384` and
`encoder5: 384→768`. The checkpoint must cover the model exactly after removing an
optional `module.` prefix. No randomly initialized encoder parameters are accepted.

In [ ]:
import hashlib, gc

DEVICE = torch.device('cuda')
model = YMamba(
    in_chans=1, num_classes=7, num_abnormal_classes=18,
    depths=[2, 2, 2, 2], feat_size=[48, 96, 192, 384], hidden_size=768,
).to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state = checkpoint['model_state_dict']
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state = checkpoint['state_dict']
elif isinstance(checkpoint, dict) and all(torch.is_tensor(v) for v in checkpoint.values()):
    state = checkpoint
else:
    raise ValueError(f'Unrecognized checkpoint structure: {type(checkpoint)}')
state = {k.removeprefix('module.'): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
print('checkpoint keys:', len(state), '| missing:', len(missing), '| unexpected:', len(unexpected))
if missing: print('missing[:20]:', missing[:20])
if unexpected: print('unexpected[:20]:', unexpected[:20])
assert not missing and not unexpected, 'Official checkpoint does not exactly cover official YMamba.'
del checkpoint, state

for p in model.parameters():
    p.requires_grad_(False)
model.eval()

def sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            block = f.read(chunk)
            if not block: break
            digest.update(block)
    return digest.hexdigest()

CHECKPOINT_SHA256 = sha256_file(CHECKPOINT_PATH)
print('checkpoint SHA256:', CHECKPOINT_SHA256)

@torch.inference_mode()
def encode_exact(x_cpu):
    x = x_cpu.to(DEVICE, non_blocking=True)
    amp_dtype = torch.float16
    with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=USE_AMP):
        stages = model.vit(x)
        hidden = model.encoder5(stages[3])
        pooled = hidden.mean(dim=(2, 3, 4))
    expected_channels = [48, 96, 192, 384]
    assert len(stages) == 4
    assert [int(t.shape[1]) for t in stages] == expected_channels
    assert hidden.ndim == 5 and hidden.shape[1] == 768
    assert pooled.shape == (x.shape[0], 768)
    assert torch.isfinite(pooled).all()
    result = {
        'pooled_encoder5': pooled.float().cpu(),
        'stage_shapes': [tuple(int(v) for v in t.shape) for t in stages],
        'encoder5_shape': tuple(int(v) for v in hidden.shape),
    }
    if SAVE_STAGE4_SPATIAL:
        result['stage4_spatial'] = stages[3].half().cpu()
    return result

print('Official EXACT Y-Mamba is frozen and ready.')

## 8. Resumable encoder loop

Outputs are written atomically. Existing files are skipped only if they contain a finite
768-d vector with the same checkpoint hash and preprocessing ID. Failures are appended to
JSONL and retried on the next run.

In [ ]:
from tqdm.auto import tqdm

RESULTS_JSONL = EXACT_MANIFEST_DIR / 'encoding_results.jsonl'
FAILURES_JSONL = EXACT_MANIFEST_DIR / 'failures.jsonl'

def output_path(volume):
    key = volume.removesuffix('.nii.gz').removesuffix('.nii')
    return EXACT_POOLED_DIR / f'{key}.pt'

def valid_existing(path):
    if not path.exists(): return False
    try:
        payload = torch.load(path, map_location='cpu')
        vector = payload['pooled_encoder5']
        return (
            tuple(vector.shape) == (768,) and torch.isfinite(vector).all().item()
            and payload.get('checkpoint_sha256') == CHECKPOINT_SHA256
            and payload.get('preprocessing', {}).get('preprocessing_id') == PREPROCESSING_ID
        )
    except Exception:
        return False

def append_jsonl(path, record):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, default=str) + '\n')

def atomic_torch_save(payload, destination):
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, destination)

def encode_targets(volumes, split):
    progress_path = EXACT_CACHE / f'progress_{split}.json'
    new = skipped = missing = failed = 0
    for i, volume in enumerate(tqdm(volumes, desc=f'EXACT {split}')):
        out = output_path(volume)
        if valid_existing(out):
            skipped += 1
            continue
        if out.exists():
            out.unlink()  # corrupt or incompatible prior result

        downloaded = download_volume(volume)
        if downloaded is None:
            missing += 1
            append_jsonl(FAILURES_JSONL, {
                'time': time.time(), 'split': split, 'volume': volume,
                'kind': 'download_missing'})
            reset_tmp()
            continue

        started = time.time()
        try:
            x, prep = preprocess_exact_nifti(downloaded)
            encoded = encode_exact(x)
            pooled = encoded['pooled_encoder5'].squeeze(0)
            payload = {
                'format_version': 1,
                'encoder': 'official_EXACT_YMamba_encoder5_global_average_pool',
                'volume': volume,
                'split': split,
                'pooled_encoder5': pooled.half().clone(),
                'stage_shapes': encoded['stage_shapes'],
                'encoder5_shape': encoded['encoder5_shape'],
                'preprocessing': prep,
                'checkpoint_name': CHECKPOINT_PATH.name,
                'checkpoint_sha256': CHECKPOINT_SHA256,
                'exact_git_remote': 'https://github.com/JasonW375/EXACT',
                'seconds': time.time() - started,
            }
            if 'stage4_spatial' in encoded:
                payload['stage4_spatial'] = encoded['stage4_spatial'].squeeze(0)
            atomic_torch_save(payload, out)
            assert valid_existing(out), f'Post-save validation failed: {out}'
            new += 1
            append_jsonl(RESULTS_JSONL, {
                'time': time.time(), 'split': split, 'volume': volume,
                'output': str(out), 'seconds': payload['seconds'],
                'stage_shapes': payload['stage_shapes'],
                'encoder5_shape': payload['encoder5_shape']})
        except Exception as exc:
            failed += 1
            print('ENCODE FAIL', volume, repr(exc))
            append_jsonl(FAILURES_JSONL, {
                'time': time.time(), 'split': split, 'volume': volume,
                'kind': 'encode_failure', 'error_type': type(exc).__name__,
                'error': str(exc)})
        finally:
            reset_tmp()
            gc.collect()
            torch.cuda.empty_cache()

        if (i + 1) % 10 == 0:
            with open(progress_path, 'w') as f:
                json.dump({'i': i + 1, 'total': len(volumes), 'new': new,
                           'skipped': skipped, 'missing': missing, 'failed': failed,
                           'updated_unix': time.time()}, f, indent=2)

    summary = {'i': len(volumes), 'total': len(volumes), 'new': new,
               'skipped': skipped, 'missing': missing, 'failed': failed,
               'updated_unix': time.time()}
    with open(progress_path, 'w') as f: json.dump(summary, f, indent=2)
    print(split, summary)
    return summary

## 9. Smoke test one validation CT

This performs a real Hugging Face download and official Y-Mamba forward pass before the
large run. It writes the first valid result to the same resumable cache.

In [ ]:
smoke_summary = encode_targets(valid_targets[:1], 'valid_smoke')
assert valid_existing(output_path(valid_targets[0])), 'Smoke-test output is invalid.'
smoke = torch.load(output_path(valid_targets[0]), map_location='cpu')
print('Smoke volume:', smoke['volume'])
print('Stage shapes:', smoke['stage_shapes'])
print('encoder5:', smoke['encoder5_shape'])
print('pooled:', tuple(smoke['pooled_encoder5'].shape), smoke['pooled_encoder5'].dtype)

## 10. Encode all validation targets

Run this cell to completion. Re-run it after any disconnect; completed outputs are
validated and skipped.

In [ ]:
valid_summary = encode_targets(valid_targets, 'valid')

## 11. Encode every longitudinal `train_*` target

This can take many Colab sessions. It encodes the 6,762 unique `train_*` volumes used by
the dated longitudinal pairs, not the entire CT-RATE training partition.

In [ ]:
train_summary = encode_targets(train_targets, 'train')

## 12. Final one-to-one coverage audit

The audit checks every longitudinal target, validates each EXACT payload, and writes a
machine-readable report to Drive.

In [ ]:
def audit(volumes):
    good, bad = [], []
    for volume in tqdm(volumes, desc='audit'):
        (good if valid_existing(output_path(volume)) else bad).append(volume)
    return good, bad

valid_good, valid_bad = audit(valid_targets)
train_good, train_bad = audit(train_targets)
audit_payload = {
    'created_unix': time.time(),
    'checkpoint_sha256': CHECKPOINT_SHA256,
    'preprocessing_id': PREPROCESSING_ID,
    'valid': {'target': len(valid_targets), 'good': len(valid_good), 'bad': valid_bad},
    'train': {'target': len(train_targets), 'good': len(train_good), 'bad': train_bad},
}
with open(EXACT_MANIFEST_DIR / 'coverage_audit.json', 'w') as f:
    json.dump(audit_payload, f, indent=2)

print('VALID:', len(valid_good), '/', len(valid_targets), '| missing/invalid:', len(valid_bad))
print('TRAIN:', len(train_good), '/', len(train_targets), '| missing/invalid:', len(train_bad))
if valid_bad: print('valid bad[:10]:', valid_bad[:10])
if train_bad: print('train bad[:10]:', train_bad[:10])
print('Audit:', EXACT_MANIFEST_DIR / 'coverage_audit.json')

## Output and downstream use

Each `.pt` payload contains a 768-d `pooled_encoder5` vector from the frozen official
EXACT image encoder. EXACT has no paired text encoder. For the temporal model, either set
`d_in=768` or learn a `Linear(768,512)` bridge before the current Difference Transformer.
The final `d_f` can still be trained against frozen CT-CLIP text prototypes, but raw EXACT
vectors are **not** already in CT-CLIP's text space.